# Stage 03: Localised (token-level) PCL detection (Part 3)

Utilises task2 dataset span text to get pure localised signals of where in paragh pcl is actually occuring, so the model can aggreagate these local signals and global context for more accurate paragraph-level predictions

Approach: Using span level PCL info to get token level PCL info
1. labels each token if they are present in positive span or not
2. Has a token classifier head that takes final token embeddings and applies linear + BCE to get token_loss for every token (averaged)
3. Takes max token PCL signal (local context) and concatenates with CLS token (global context)
4. Linear + BCE on concatenated token to get paragraph_loss
5. Total_loss = paragraph_loss  +  lambda * token_loss

## Imports & Dataset utilities

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
import os
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers import get_linear_schedule_with_warmup

# project root (one level above notebooks)
ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

from src.data.make_dataset import build_task1_task2_with_spans, validate_span_ranges, validate_span_text_alignment, truncation_rate_by_label
from src.training.metrics import stats_on_loader
from src.training.tokenization_utils import ensure_token_cache, TensorCacheDataset, PCLTokenDataset

RAW_TASK1 = ROOT / "data" / "raw" / "dontpatronizeme_pcl.tsv"
RAW_TASK2 = ROOT / "data" / "raw" / "dontpatronizeme_categories.tsv"
TRAIN_SPLIT = ROOT / "data" / "splits" / "train_semeval_parids-labels.csv"
DEV_SPLIT = ROOT / "data" / "splits" / "dev_semeval_parids-labels.csv"

train_df, dev_df, pcl_df, spans_df_norm = build_task1_task2_with_spans(
    raw_task1_path=RAW_TASK1,
    raw_task2_path=RAW_TASK2,
    train_split_path=TRAIN_SPLIT,
    dev_split_path=DEV_SPLIT,
)

print("train_df:", train_df.shape, "| positives:", int(train_df["label_bin"].sum()))
print("dev_df:", dev_df.shape, "| positives:", int(dev_df["label_bin"].sum()))
print("Example span_ranges:", train_df.loc[train_df["label_bin"].idxmax(), "span_ranges"] if len(train_df) else None)

/home/joshua_killa/.pyenv/versions/pcl-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train_df: (8375, 8) | positives: 794
dev_df: (2094, 8) | positives: 199
Example span_ranges: [(157, 242)]


In [2]:
# Make sure span ranges are valid
validate_span_ranges(train_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="train")
validate_span_ranges(dev_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="dev")

# Check task2 spans match the corresponding substrings in task1 (only 2 mismatches and only by one charcter offset which is acceptable given token based stuff)
validate_span_text_alignment(
    train_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="train", max_mismatches=1
)
validate_span_text_alignment(
    dev_df, pcl_df=pcl_df, spans_df_norm=spans_df_norm, name="dev", max_mismatches=1
)


== validate_span_ranges: train ==
shape: (8375, 8)
missing (par_id,s,e) pairs: 0
out-of-bounds (par_id,s,e,L): 0
negatives with spans: 0

== validate_span_ranges: dev ==
shape: (2094, 8)
missing (par_id,s,e) pairs: 0
out-of-bounds (par_id,s,e,L): 0
negatives with spans: 0

== validate_span_text_alignment: train ==
total spans checked: 2466
text mismatches: 1
examples:


,par_id,span_start_norm,span_finish_norm,span_text,task1_substr
1134,4655,16,114,"To suffer and empathize together with others ,...","o suffer and empathize together with others , ..."



== validate_span_text_alignment: dev ==
total spans checked: 714
text mismatches: 1
examples:


,par_id,span_start_norm,span_finish_norm,span_text,task1_substr
482,6708,0,133,I only wish they can one day wake up to realis...,I only wish they can one day wake up to relise...


In [3]:
# ---- choose backbone here ----
MODEL_CANDIDATES = {
    "deberta": "microsoft/deberta-v3-base",
    "albert": "albert-base-v2",
    "albert_large": "albert-large-v2",
}

MODEL_KEY = os.getenv("PCL_BACKBONE", "albert_large")  # set to "albert" to try ALBERT
MODEL_NAME = MODEL_CANDIDATES[MODEL_KEY]
MAX_LEN = 192

def load_tokenizer(model_name: str):
    # Online first, then local cache fallback (for DNS/no-internet issues)
    try:
        return AutoTokenizer.from_pretrained(model_name, use_fast=True)
    except Exception as e:
        print(f"Tokenizer load failed ({type(e).__name__}: {e}). Trying local cache...")
        return AutoTokenizer.from_pretrained(model_name, use_fast=True, local_files_only=True)

tokenizer = load_tokenizer(MODEL_NAME)



In [4]:
# Verify max token length covers most examples (192 seems fine)
print(truncation_rate_by_label(tokenizer, train_df, 192))
print(truncation_rate_by_label(tokenizer, train_df, 200))
print(truncation_rate_by_label(tokenizer, train_df, 224))

{'max_len': 192, 'all_truncated_pct': 0.6447761194029851, 'pos_truncated_pct': 0.5037783375314862, 'neg_truncated_pct': 0.6595435958316844, 'pos_p99_len': 175, 'neg_p99_len': 180}
{'max_len': 200, 'all_truncated_pct': 0.4895522388059702, 'pos_truncated_pct': 0.5037783375314862, 'neg_truncated_pct': 0.4880622609154465, 'pos_p99_len': 175, 'neg_p99_len': 180}
{'max_len': 224, 'all_truncated_pct': 0.20298507462686569, 'pos_truncated_pct': 0.3778337531486146, 'neg_truncated_pct': 0.18467220683287164, 'pos_p99_len': 175, 'neg_p99_len': 180}


In [4]:
# How to pool token-level logits to single paragraph level logit (used in TokenCLSModel)

class LogitPooler(nn.Module):
    """
    Pool token-level logits (B,T) -> (B,1) using a boolean mask (B,T).

    Modes:
      - "max": masked max over tokens
      - "topk_mean": mean of top-k masked logits (k set by top_k)
    """
    def __init__(self, mode: str = "max", top_k: int = 3, sentinel: float = -1e4):
        super().__init__()
        self.mode = str(mode)
        self.top_k = int(top_k)
        self.sentinel = float(sentinel)

        if self.mode not in {"max", "topk_mean"}:
            raise ValueError(f"Unknown pooling mode: {self.mode}")

    def forward(self, token_logits: torch.Tensor, token_mask: torch.Tensor) -> torch.Tensor:
        """
        token_logits: (B,T) float
        token_mask:   (B,T) bool (True = keep / real tokens)
        returns:      (B,1)
        """
        token_mask = token_mask.to(dtype=torch.bool)

        # Edge-case guard: if a sample has no real tokens, return 0.0 (neutral feature)
        no_real = ~token_mask.any(dim=1, keepdim=True)  # (B,1)

        x = token_logits.masked_fill(~token_mask, self.sentinel)  # (B,T)

        if self.mode == "max":
            pooled = x.max(dim=1, keepdim=True).values  # (B,1)
            pooled = pooled.masked_fill(no_real, 0.0)
            return pooled

        # self.mode == "topk_mean"
        B, T = x.shape
        k = min(self.top_k, T)
        topk_vals = torch.topk(x, k=k, dim=1).values  # (B,k)

        # If k > #real tokens, topk will include sentinel values; exclude them from the mean.
        valid = topk_vals > (self.sentinel + 1.0)
        denom = valid.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (topk_vals.masked_fill(~valid, 0.0).sum(dim=1, keepdim=True)) / denom
        pooled = pooled.masked_fill(no_real, 0.0)
        return pooled

In [5]:
class TokenCLSModel(nn.Module):
    def __init__(self, model_name, lambda_token=0.3, pool_mode="max", top_k=3, alpha_neg=0.25):
        super().__init__()

        # Online first, then local cache fallback
        try:
            self.encoder = AutoModel.from_pretrained(model_name)
        except Exception as e:
            print(f"Model load failed ({type(e).__name__}: {e}). Trying local cache...")
            self.encoder = AutoModel.from_pretrained(model_name, local_files_only=True)

        hidden = self.encoder.config.hidden_size

        self.token_head = nn.Linear(hidden, 1)
        self.paragraph_head = nn.Linear(hidden + 1, 1)

        self.lambda_token = float(lambda_token)
        self.alpha_neg = float(alpha_neg)

        # modular pooling (swap "max" <-> "topk_mean")
        self.pooler = LogitPooler(mode=pool_mode, top_k=top_k)

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
        token_labels=None,
        token_loss_mask=None,
        paragraph_label=None,
    ):
        # Some backbones ignore token_type_ids; some accept it.
        # Try passing it; if unsupported, fall back.
        try:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
        except TypeError:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        hidden = outputs.last_hidden_state  # (B,T,H)

        # keep dtype consistent with heads (prevents dtype mismatch issues)
        hidden = hidden.to(dtype=self.token_head.weight.dtype)

        cls = hidden[:, 0]  # (B,H)
        token_logits = self.token_head(hidden).squeeze(-1)  # (B,T)

        # safe token mask for pooling + token-loss
        if token_loss_mask is None:
            token_loss_mask = attention_mask == 1
        token_loss_mask = token_loss_mask.to(dtype=torch.bool)

        pooled = self.pooler(token_logits, token_loss_mask)  # (B,1)

        fused = torch.cat([cls, pooled], dim=1)  # (B,H+1)
        paragraph_logit = self.paragraph_head(fused).squeeze(-1)  # (B,)

        loss_par = None
        if paragraph_label is not None:
            loss_par = F.binary_cross_entropy_with_logits(
                paragraph_logit.float(),
                paragraph_label.float(),
            )

        # Compute token loss independently (enables true token-only warm start)
        loss_tok = None
        if token_labels is not None:
            per_tok = F.binary_cross_entropy_with_logits(
                token_logits.float(),
                token_labels.float(),
                reduction="none",
            )  # (B,T)

            # downweight negatives globally (token_labels==0)
            weights = torch.where(token_labels > 0.5, 1.0, self.alpha_neg).to(per_tok.dtype)  # (B,T)
            per_tok = per_tok * weights

            per_tok = per_tok.masked_fill(~token_loss_mask, 0.0)
            denom = token_loss_mask.sum().clamp(min=1).to(per_tok.dtype)
            loss_tok = per_tok.sum() / denom

        loss = None
        if (loss_par is not None) and (loss_tok is not None):
            loss = loss_par + self.lambda_token * loss_tok
        elif loss_par is not None:
            loss = loss_par
        elif loss_tok is not None:
            loss = self.lambda_token * loss_tok  # for warm start set lambda_token=1.0

        return {
            "loss": loss,
            "loss_par": loss_par,
            "loss_tok": loss_tok,
            "paragraph_logit": paragraph_logit,
            "token_logits": token_logits,
        }

In [6]:
CACHE_DIR = ROOT / "data" / "cache" / f"pcl_tok_{MODEL_KEY}_len{MAX_LEN}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_cache = ensure_token_cache(
    train_df,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    cache_path=CACHE_DIR / "train.pt",
    model_key=MODEL_KEY,
    model_name=MODEL_NAME,
)
dev_cache = ensure_token_cache(
    dev_df,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    cache_path=CACHE_DIR / "dev.pt",
    model_key=MODEL_KEY,
    model_name=MODEL_NAME,
)

train_dataset = TensorCacheDataset(train_cache)
dev_dataset = TensorCacheDataset(dev_cache)

print("train_dataset:", type(train_dataset), "len=", len(train_dataset))
print("dev_dataset:", type(dev_dataset), "len=", len(dev_dataset))

train_dataset: <class 'src.training.tokenization_utils.TensorCacheDataset'> len= 8375
dev_dataset: <class 'src.training.tokenization_utils.TensorCacheDataset'> len= 2094


In [7]:
from torch.optim import AdamW
from tqdm import tqdm
import torch
import numpy as np
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, WeightedRandomSampler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _amp_settings(model_key: str, model_name: str):
    key = (model_key or "").lower()
    name = (model_name or "").lower()

    if ("albert" in key) or ("albert" in name):
        enabled = torch.cuda.is_available()
        return enabled, torch.float16, True  # FP16 + GradScaler

    if ("deberta" in key) or ("deberta" in name):
        enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        return enabled, torch.bfloat16, False  # BF16, no GradScaler

    return False, None, False

USE_AMP, AMP_DTYPE, NEEDS_SCALER = _amp_settings(MODEL_KEY, MODEL_NAME)
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))

print(
    f"DEVICE={DEVICE} | backbone={MODEL_KEY} | amp={USE_AMP} | amp_dtype={AMP_DTYPE} | "
    f"scaler={bool(scaler.is_enabled())}"
)

model = TokenCLSModel(MODEL_NAME, lambda_token=0.3, pool_mode="max").to(DEVICE)

# train_dataset = PCLTokenDataset(train_df)

# ---- balanced paragraph sampling (approx 50/50 pos/neg) ----
labels = train_df["label_bin"].astype(int).to_numpy()
pos = int(labels.sum())
neg = int(len(labels) - pos)

w_pos = 1.0 / max(pos, 1)
w_neg = 1.0 / max(neg, 1)

sample_weights = torch.tensor([w_pos if y == 1 else w_neg for y in labels], dtype=torch.double)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),  # keep "epoch" size comparable to dataset size
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=sampler,   # <- sampler replaces shuffle
)

# separate loaders for F1 computation (no shuffle)
train_eval_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
# dev_dataset = PCLTokenDataset(dev_df)
dev_eval_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

optimizer = AdamW(
    model.parameters(),
    lr=7e-6,
    weight_decay=1e-2,
)

THRESH = 0.5
GRAD_ACCUM = 2  # <- match your optuna best configs; effective batch = 16 * 2 = 32
EPOCHS = 7

total_update_steps = (len(train_loader) * EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM
warmup_steps = int(0.07 * total_update_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)



/tmp/ipykernel_1080127/1639145740.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))


DEVICE=cuda | backbone=albert_large | amp=True | amp_dtype=torch.float16 | scaler=True


Loading weights: 100%|██████████| 25/25 [00:00<00:00, 436.21it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
b = next(iter(train_loader))
print("batch pos rate:", float(b["paragraph_label"].mean()))

batch pos rate: 0.5625


In [8]:
# --- Optuna: setup (NEW CELL) ---
import optuna
from optuna.pruners import MedianPruner
import random, gc

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

OPTUNA_DIR = ROOT / "runs" / "optuna_stage04_localised_tokencls"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

storage_url = f"sqlite:///{(OPTUNA_DIR / 'study.db').as_posix()}"
study = optuna.create_study(
    study_name="stage04_localised_tokencls",
    direction="maximize",
    storage=storage_url,
    load_if_exists=True,
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=1, interval_steps=1),
)

print("storage:", storage_url)
print("trials so far:", len(study.trials))

[I 2026-03-04 15:16:23,387] Using an existing study with name 'stage04_localised_tokencls' instead of creating a new one.


storage: sqlite:////home/joshua_killa/doc/y3/PCL-detection/runs/optuna_stage04_localised_tokencls/study.db
trials so far: 23


In [10]:
# --- Optuna: objective (NEW CELL) ---
from optuna import trial
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim import AdamW

# fixed knobs (match your notebook)
EPOCHS = 7
BATCH_SIZE = 16
GRAD_ACCUM = 2
THRESH = 0.5
TOKEN_ONLY_EPOCHS = 1
EARLY_STOP_PATIENCE = 4

# reuse the same balancing weights you already computed from train_df
_labels = train_df["label_bin"].astype(int).to_numpy()
_pos = int(_labels.sum())
_neg = int(len(_labels) - _pos)
_w_pos = 1.0 / max(_pos, 1)
_w_neg = 1.0 / max(_neg, 1)
_sample_weights = torch.tensor([_w_pos if y == 1 else _w_neg for y in _labels], dtype=torch.double)

def objective(trial: optuna.Trial):
    set_seed(SEED)

    # requested search params
    alpha_neg = trial.suggest_float("alpha_neg", 0.1, 0.7)
    lambda_token = trial.suggest_float("lambda_token", 0.05, 1.0)
    token_only_epochs = trial.suggest_int("token_only_epochs", 0, 3)

    # extra (useful) params
    lr = trial.suggest_float("lr", 2e-6, 3e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    trial.set_user_attr("model_name", MODEL_NAME)
    trial.set_user_attr("model_key", MODEL_KEY)
    trial.set_user_attr("max_len", int(MAX_LEN))
    trial.set_user_attr("amp", bool(USE_AMP))
    trial.set_user_attr("amp_dtype", str(AMP_DTYPE) if AMP_DTYPE is not None else None)
    trial.set_user_attr("grad_accum", int(GRAD_ACCUM))
    trial.set_user_attr("batch_size", int(BATCH_SIZE))
    trial.set_user_attr("epochs", int(EPOCHS))

    sampler = WeightedRandomSampler(_sample_weights, num_samples=len(_sample_weights), replacement=True)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
    train_eval_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
    dev_eval_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

    model = TokenCLSModel(
        MODEL_NAME,
        lambda_token=lambda_token,
        pool_mode="max",
        alpha_neg=alpha_neg,
    ).to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_update_steps = (len(train_loader) * EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM
    warmup_steps = int(0.07 * total_update_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )

    local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))

    best_f1 = -1.0
    no_improve = 0

    try:
        for epoch in range(EPOCHS):
            token_only = epoch < token_only_epochs

            # curriculum
            if token_only:
                for p in model.paragraph_head.parameters():
                    p.requires_grad = False
                model.lambda_token = 1.0
            else:
                for p in model.paragraph_head.parameters():
                    p.requires_grad = True
                model.lambda_token = float(lambda_token)

            model.train()
            optimizer.zero_grad(set_to_none=True)

            last_step = 0
            for step, batch in enumerate(train_loader, start=1):
                last_step = step
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                if token_only:
                    batch["paragraph_label"] = None

                if USE_AMP:
                    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                        out = model(**batch)
                        loss = out["loss"]
                else:
                    out = model(**batch)
                    loss = out["loss"]

                loss = loss / GRAD_ACCUM

                if local_scaler.is_enabled():
                    local_scaler.scale(loss).backward()
                else:
                    loss.backward()

                if (step % GRAD_ACCUM) == 0:
                    if local_scaler.is_enabled():
                        local_scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                    if local_scaler.is_enabled():
                        local_scaler.step(optimizer)
                        local_scaler.update()
                    else:
                        optimizer.step()

                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

            # flush remainder
            if last_step and (last_step % GRAD_ACCUM) != 0:
                if local_scaler.is_enabled():
                    local_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                if local_scaler.is_enabled():
                    local_scaler.step(optimizer)
                    local_scaler.update()
                else:
                    optimizer.step()

                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            # eval: dev f1 is the Optuna score
            dev_stats = stats_on_loader(
                dev_eval_loader,
                model,
                DEVICE,
                USE_AMP,
                AMP_DTYPE,
                threshold=THRESH,
                compute_token_loss=True,
                compute_token_metrics=True,
                limit_batches=None,
            )
            dev_f1 = float(dev_stats["f1"])

            train_stats = stats_on_loader(
                train_eval_loader,
                model,
                DEVICE,
                USE_AMP,
                AMP_DTYPE,
                threshold=THRESH,
                compute_token_loss=True,
                compute_token_metrics=True,
                limit_batches=20,  # quick estimate like your manual loop
            )

            print(
                f"[trial {trial.number}] Epoch {epoch} | "
                f"train loss={train_stats['loss']:.4f} (par={train_stats['loss_par']:.4f}, tok={train_stats['loss_tok']}) | "
                f"train f1={train_stats['f1']:.4f} acc={train_stats['acc']:.4f} | "
                f"train tok_acc={train_stats.get('tok_acc')} | "
                f"dev loss={dev_stats['loss']:.4f} (par={dev_stats['loss_par']:.4f}, tok={dev_stats['loss_tok']}) | "
                f"dev f1={dev_stats['f1']:.4f} acc={dev_stats['acc']:.4f} | "
                f"dev tok_acc={dev_stats.get('tok_acc')}"
            )

            trial.set_user_attr(f"epoch_{epoch}_train", {k: float(v) for k, v in train_stats.items() if isinstance(v, (int, float, np.floating))})
            trial.set_user_attr(f"epoch_{epoch}_dev",   {k: float(v) for k, v in dev_stats.items() if isinstance(v, (int, float, np.floating))})

            # track best epoch too
            if dev_f1 >= best_f1:
                trial.set_user_attr("best_epoch", int(epoch))
                trial.set_user_attr("best_dev_stats", {k: float(v) for k, v in dev_stats.items() if isinstance(v, (int, float, np.floating))})
                trial.set_user_attr("best_train_stats", {k: float(v) for k, v in train_stats.items() if isinstance(v, (int, float, np.floating))})

            trial.report(dev_f1, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            if dev_f1 > best_f1 + 1e-6:
                best_f1 = dev_f1
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= EARLY_STOP_PATIENCE:
                    break

        return float(best_f1)

    except torch.cuda.OutOfMemoryError:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise optuna.TrialPruned()
    finally:
        del model, optimizer, scheduler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [11]:
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import numpy as np

def evaluate(model, dataset):
    loader = DataLoader(dataset, batch_size=32)
    model.eval()

    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            all_logits.append(out["paragraph_logit"].detach().float().cpu())
            all_labels.append(batch["paragraph_label"].detach().float().cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)

    probs = 1.0 / (1.0 + np.exp(-logits))

    best_f1 = 0.0
    best_thresh = 0.5
    for t in np.linspace(0.1, 0.9, 81):
        preds = (probs > t).astype(int)
        f1 = f1_score(labels, preds, pos_label=1)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = float(t)

    return best_f1, best_thresh

In [12]:
N_TRIALS = 20
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

print("best value (dev f1):", study.best_value)
print("best params:", study.best_params)

Loading weights: 100%|██████████| 25/25 [00:00<00:00, 418.91it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_198924/1146678584.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))


[trial 0] Epoch 0 | train loss=0.2024 (par=0.1925, tok=0.19684407413005828) | train f1=0.5806 acc=0.9289 | train tok_acc=0.8670465092296586 | dev loss=0.2546 (par=0.2447, tok=0.19727026784058774) | dev f1=0.5108 acc=0.9131 | dev tok_acc=0.8640160511014658
[trial 0] Epoch 1 | train loss=0.2318 (par=0.2245, tok=0.14428179785609246) | train f1=0.6506 acc=0.9094 | train tok_acc=0.8808007631226152 | dev loss=0.3444 (par=0.3364, tok=0.15880505492289862) | dev f1=0.5331 acc=0.8787 | dev tok_acc=0.8633035787404799
[trial 0] Epoch 2 | train loss=0.1485 (par=0.1458, tok=0.053772273007780313) | train f1=0.8352 acc=0.9664 | train tok_acc=0.9539032690522842 | dev loss=0.4413 (par=0.4355, tok=0.11621331717028763) | dev f1=0.5756 acc=0.9169 | dev tok_acc=0.9241012202112849
[trial 0] Epoch 3 | train loss=0.1427 (par=0.1405, tok=0.04369441764429212) | train f1=0.8417 acc=0.9680 | train tok_acc=0.9583376301949056 | dev loss=0.5168 (par=0.5098, tok=0.1384563403570968) | dev f1=0.5556 acc=0.9083 | dev tok

[I 2026-03-01 08:31:02,501] Trial 0 finished with value: 0.5756097560975609 and parameters: {'alpha_neg': 0.37377316102499947, 'lambda_token': 0.050516984494784874, 'token_only_epochs': 0, 'lr': 2.298872716131426e-05, 'weight_decay': 0.06649348493890288}. Best is trial 0 with value: 0.5756097560975609.
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 316.22it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect

[trial 1] Epoch 0 | train loss=0.5693 (par=0.4373, tok=0.13196820467710496) | train f1=0.0000 acc=0.9141 | train tok_acc=0.9059502939053315 | dev loss=0.5813 (par=0.4378, tok=0.14345913323940654) | dev f1=0.0000 acc=0.9050 | dev tok_acc=0.8990418475145361
[trial 1] Epoch 1 | train loss=0.3233 (par=0.2580, tok=0.10326835233718157) | train f1=0.6225 acc=0.8977 | train tok_acc=0.920555326389605 | dev loss=0.4241 (par=0.3344, tok=0.14172036573290825) | dev f1=0.5359 acc=0.8734 | dev tok_acc=0.8973876013430514
[trial 1] Epoch 2 | train loss=0.1793 (par=0.1495, tok=0.04705766402184963) | train f1=0.8000 acc=0.9578 | train tok_acc=0.97046767041353 | dev loss=0.4088 (par=0.3185, tok=0.14267244988657307) | dev f1=0.5893 acc=0.9121 | dev tok_acc=0.9346818442388011
[trial 1] Epoch 3 | train loss=0.0912 (par=0.0762, tok=0.023766176775097847) | train f1=0.8898 acc=0.9789 | train tok_acc=0.9856141074559142 | dev loss=0.4636 (par=0.3524, tok=0.1757893729954958) | dev f1=0.5618 acc=0.9255 | dev tok_ac

[I 2026-03-01 09:45:22,780] Trial 1 finished with value: 0.5898123324396782 and parameters: {'alpha_neg': 0.5330674584923091, 'lambda_token': 0.6326810817319789, 'token_only_epochs': 1, 'lr': 1.5639475270427764e-05, 'weight_decay': 0.0074233859691767885}. Best is trial 1 with value: 0.5898123324396782.
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 347.85it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect

...

[trial 17] Epoch 0 | train loss=0.7193 (par=0.4618, tok=0.25749257802963255) | train f1=0.0000 acc=0.9125 | train tok_acc=0.8174822109930906 | dev loss=0.7209 (par=0.4686, tok=0.2522452636198564) | dev f1=0.0000 acc=0.9045 | dev tok_acc=0.8254688395708787
[trial 17] Epoch 1 | train loss=1.0092 (par=0.7256, tok=0.29558274894952774) | train f1=0.1583 acc=0.0859 | train tok_acc=0.9433845519232752 | dev loss=1.0072 (par=0.7248, tok=0.2943343050552137) | dev f1=0.1737 acc=0.0960 | dev tok_acc=0.9404143804766194


[I 2026-03-01 23:55:33,613] Trial 17 pruned. 
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 394.34it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_198924/1146678584.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCAL

[trial 18] Epoch 0 | train loss=0.5550 (par=0.4287, tok=0.1263172823935747) | train f1=0.0000 acc=0.9141 | train tok_acc=0.8650484685985357 | dev loss=0.5684 (par=0.4365, tok=0.13182568380778487) | dev f1=0.0000 acc=0.9040 | dev tok_acc=0.8599295716976496
[trial 18] Epoch 1 | train loss=0.5218 (par=0.4394, tok=0.08242329675704241) | train f1=0.0000 acc=0.9141 | train tok_acc=0.91039754563267 | dev loss=0.5636 (par=0.4454, tok=0.11819264213695671) | dev f1=0.0000 acc=0.9050 | dev tok_acc=0.8890426664482843


[I 2026-03-02 00:15:50,091] Trial 18 pruned. 
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 409.22it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_198924/1146678584.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCAL

[trial 19] Epoch 0 | train loss=0.3060 (par=0.2399, tok=0.15024186633527278) | train f1=0.5368 acc=0.9016 | train tok_acc=0.9273228833659894 | dev loss=0.3280 (par=0.2600, tok=0.15419122181607015) | dev f1=0.5307 acc=0.8978 | dev tok_acc=0.9228236835639997
[trial 19] Epoch 1 | train loss=0.3459 (par=0.2829, tok=0.14289570413529873) | train f1=0.5699 acc=0.8727 | train tok_acc=0.904377642569867 | dev loss=0.4069 (par=0.3363, tok=0.16011692222320673) | dev f1=0.4992 acc=0.8496 | dev tok_acc=0.8931455245270657


In [ ]:
# #empty torch cahce

# torch.cuda.empty_cache()

# Retrain Best Model & Generate Submission Files

Hyperparameters are pulled directly from the Optuna study's best trial.

In [8]:
# ============================================================
# DETERMINISTIC RETRAINING WITH BEST TRIAL HYPERPARAMETERS
# ============================================================
# Pulls hyperparams directly from the Optuna study
# IMPORTANT: Must match the objective() function EXACTLY for reproducibility

import random
import gc

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Load best trial from study
best_trial = study.best_trial

# === PULL HYPERPARAMETERS FROM OPTUNA ===
SEED = 42
ALPHA_NEG = best_trial.params["alpha_neg"]
LAMBDA_TOKEN = best_trial.params["lambda_token"]
LR = best_trial.params["lr"]
TOKEN_ONLY_EPOCHS = best_trial.params["token_only_epochs"]
WEIGHT_DECAY = best_trial.params["weight_decay"]

# Get best epoch from user_attrs (0-indexed)
BEST_EPOCH = best_trial.user_attrs.get("best_epoch", 5)
EPOCHS_TO_TRAIN = BEST_EPOCH + 1  # Train until best epoch (inclusive)

# Fixed hyperparameters - MUST match objective() exactly
BATCH_SIZE = best_trial.user_attrs.get("batch_size", 16)
GRAD_ACCUM = best_trial.user_attrs.get("grad_accum", 2)
THRESH = 0.5

# CRITICAL: Original objective used EPOCHS=7 for scheduler, not EPOCHS_TO_TRAIN
# We must use the same value to get identical LR schedule
ORIGINAL_EPOCHS = best_trial.user_attrs.get("epochs", 7)

print("=" * 60)
print("LOADING HYPERPARAMETERS FROM OPTUNA STUDY")
print(f"Best Trial: {best_trial.number} | F1: {best_trial.value:.4f}")
print(f"SEED={SEED}")
print(f"ALPHA_NEG={ALPHA_NEG:.4f}")
print(f"LAMBDA_TOKEN={LAMBDA_TOKEN:.4f}")
print(f"LR={LR:.2e}")
print(f"TOKEN_ONLY_EPOCHS={TOKEN_ONLY_EPOCHS}")
print(f"WEIGHT_DECAY={WEIGHT_DECAY:.4f}")
print(f"BEST_EPOCH={BEST_EPOCH} -> EPOCHS_TO_TRAIN={EPOCHS_TO_TRAIN}")
print(f"ORIGINAL_EPOCHS={ORIGINAL_EPOCHS} (used for scheduler)")
print(f"BATCH_SIZE={BATCH_SIZE} | GRAD_ACCUM={GRAD_ACCUM}")
print("=" * 60)

# ============================================================
# MUST MATCH objective() EXACTLY FROM HERE
# ============================================================

# Set seed FIRST (same position as in objective)
set_seed(SEED)

# Same sampler setup as objective (uses global _sample_weights defined earlier)
# Recompute to ensure consistency
_labels = train_df["label_bin"].astype(int).to_numpy()
_pos = int(_labels.sum())
_neg = int(len(_labels) - _pos)
_w_pos = 1.0 / max(_pos, 1)
_w_neg = 1.0 / max(_neg, 1)
_sample_weights = torch.tensor([_w_pos if y == 1 else _w_neg for y in _labels], dtype=torch.double)

sampler = WeightedRandomSampler(_sample_weights, num_samples=len(_sample_weights), replacement=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
train_eval_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
dev_eval_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

# Initialize model with exact hyperparams
best_model = TokenCLSModel(
    MODEL_NAME,
    lambda_token=LAMBDA_TOKEN,
    pool_mode="max",
    alpha_neg=ALPHA_NEG,
).to(DEVICE)

optimizer = AdamW(best_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# CRITICAL: Use ORIGINAL_EPOCHS (7) for scheduler, not EPOCHS_TO_TRAIN
# This matches the objective() which always computed schedule for full EPOCHS
total_update_steps = (len(train_loader) * ORIGINAL_EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM
warmup_steps = int(0.07 * total_update_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)

local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))

print(f"total_update_steps={total_update_steps} | warmup_steps={warmup_steps}")

# Track best model by dev F1
best_f1_so_far = -1.0
best_model_state = None
best_epoch_actual = -1

# Training loop - matches objective() exactly
for epoch in range(EPOCHS_TO_TRAIN):
    token_only = epoch < TOKEN_ONLY_EPOCHS
    
    # Curriculum: token-only warmup (same logic as objective)
    if token_only:
        for p in best_model.paragraph_head.parameters():
            p.requires_grad = False
        best_model.lambda_token = 1.0
    else:
        for p in best_model.paragraph_head.parameters():
            p.requires_grad = True
        best_model.lambda_token = float(LAMBDA_TOKEN)
    
    best_model.train()
    optimizer.zero_grad(set_to_none=True)
    
    last_step = 0
    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}"), start=1):
        last_step = step
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        if token_only:
            batch["paragraph_label"] = None
        
        if USE_AMP:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                out = best_model(**batch)
                loss = out["loss"]
        else:
            out = best_model(**batch)
            loss = out["loss"]
        
        loss = loss / GRAD_ACCUM
        
        if local_scaler.is_enabled():
            local_scaler.scale(loss).backward()
        else:
            loss.backward()
        
        if (step % GRAD_ACCUM) == 0:
            if local_scaler.is_enabled():
                local_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(best_model.parameters(), 1.0)
            
            if local_scaler.is_enabled():
                local_scaler.step(optimizer)
                local_scaler.update()
            else:
                optimizer.step()
            
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
    
    # Flush remainder (same as objective)
    if last_step and (last_step % GRAD_ACCUM) != 0:
        if local_scaler.is_enabled():
            local_scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(best_model.parameters(), 1.0)
        
        if local_scaler.is_enabled():
            local_scaler.step(optimizer)
            local_scaler.update()
        else:
            optimizer.step()
        
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
    
    # Evaluate each epoch
    dev_stats = stats_on_loader(
        dev_eval_loader, best_model, DEVICE, USE_AMP, AMP_DTYPE,
        threshold=THRESH, compute_token_loss=True, compute_token_metrics=True, limit_batches=None
    )
    train_stats = stats_on_loader(
        train_eval_loader, best_model, DEVICE, USE_AMP, AMP_DTYPE,
        threshold=THRESH, compute_token_loss=True, compute_token_metrics=True, limit_batches=20
    )
    is_best = dev_stats['f1'] > best_f1_so_far
    print(
        f"Epoch {epoch} | "
        f"train loss={train_stats['loss']:.4f} f1={train_stats['f1']:.4f} | "
        f"dev loss={dev_stats['loss']:.4f} f1={dev_stats['f1']:.4f}"
        f"{' <-- BEST' if is_best else ''}"
    )
    
    # Save best model state (by dev F1)
    if is_best:
        best_f1_so_far = dev_stats['f1']
        best_model_state = {k: v.cpu().clone() for k, v in best_model.state_dict().items()}
        best_epoch_actual = epoch

# Restore best model
best_model.load_state_dict(best_model_state)

print("\\n" + "=" * 60)
print(f"TRAINING COMPLETE - Best model from epoch {best_epoch_actual}")
print(f"Best F1: {best_f1_so_far:.4f} (expected: {best_trial.value:.4f})")
print("=" * 60)

# ============================================================
# SAVE MODEL CHECKPOINT
# ============================================================
CHECKPOINT_DIR = ROOT / "experiments" / "token-level"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = CHECKPOINT_DIR / "best_model.pt"
torch.save({
    "model_state_dict": best_model.state_dict(),
    "epoch": best_epoch_actual,
    "best_f1": best_f1_so_far,
    "hyperparams": {
        "alpha_neg": ALPHA_NEG,
        "lambda_token": LAMBDA_TOKEN,
        "lr": LR,
        "token_only_epochs": TOKEN_ONLY_EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "original_epochs": ORIGINAL_EPOCHS,
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
    },
}, checkpoint_path)
print(f"\\nCheckpoint saved to: {checkpoint_path}")

LOADING HYPERPARAMETERS FROM OPTUNA STUDY
Best Trial: 1 | F1: 0.5898
SEED=42
ALPHA_NEG=0.5331
LAMBDA_TOKEN=0.6327
LR=1.56e-05
TOKEN_ONLY_EPOCHS=1
WEIGHT_DECAY=0.0074
BEST_EPOCH=5 -> EPOCHS_TO_TRAIN=6
ORIGINAL_EPOCHS=7 (used for scheduler)
BATCH_SIZE=16 | GRAD_ACCUM=2


Loading weights: 100%|██████████| 25/25 [00:00<00:00, 433.82it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_1066364/1178679244.py:96: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))


total_update_steps=1834 | warmup_steps=128


Epoch 0: 100%|██████████| 524/524 [05:30<00:00,  1.58it/s]


Epoch 0 | train loss=0.5693 f1=0.0000 | dev loss=0.5813 f1=0.0000 <-- BEST


...

Epoch 3 | train loss=0.1076 f1=0.8745 | dev loss=0.4771 f1=0.5464


Epoch 4: 100%|██████████| 524/524 [05:28<00:00,  1.59it/s]


Epoch 4 | train loss=0.1137 f1=0.8971 | dev loss=0.5520 f1=0.5749


Epoch 5: 100%|██████████| 524/524 [05:22<00:00,  1.62it/s]


Epoch 5 | train loss=0.0672 f1=0.9402 | dev loss=0.5872 f1=0.5613
\n============================================================
TRAINING COMPLETE - Best model from epoch 2
Best F1: 0.5797 (expected: 0.5898)
\nCheckpoint saved to: /home/joshua_killa/doc/y3/PCL-detection/experiments/token-level/best_model.pt


In [9]:
# ============================================================
# GENERATE SUBMISSION FILES (dev.txt, test.txt)
# ============================================================

from src.data.make_dataset import load_test_dataset
from torch.utils.data import DataLoader, TensorDataset

# Load test data
test_df = load_test_dataset()
print(f"Test set: {len(test_df)} paragraphs")

# Output directory
OUTPUT_DIR = ROOT / "experiments" / "token-level"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load model from checkpoint (avoids kernel restart issues)
checkpoint_path = OUTPUT_DIR / "best_model.pt"
if checkpoint_path.exists():
    print(f"Loading model from checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    best_model = TokenCLSModel(
        MODEL_NAME,
        lambda_token=checkpoint["hyperparams"]["lambda_token"],
        pool_mode="max",
        alpha_neg=checkpoint["hyperparams"]["alpha_neg"],
    ).to(DEVICE)
    best_model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded model from epoch {checkpoint['epoch']} with F1={checkpoint['best_f1']:.4f}")
else:
    print("No checkpoint found, using model from training")

# Inline prediction function (avoids cached import issue)
def _get_preds(model, df, tokenizer, max_len, device, batch_size=32, use_amp=False, amp_dtype=None, threshold=0.5):
    model.eval()
    texts = df["text"].fillna("").astype(str).tolist()
    encodings = tokenizer(texts, max_length=max_len, padding="max_length", truncation=True, return_tensors="pt")
    dataset = TensorDataset(
        encodings["input_ids"],
        encodings["attention_mask"],
        encodings.get("token_type_ids", torch.zeros_like(encodings["input_ids"])),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            input_ids, attention_mask, token_type_ids = [b.to(device) for b in batch]
            if use_amp and amp_dtype:
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            logits = outputs["paragraph_logit"]
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            preds = (probs >= threshold).astype(int).tolist()
            all_preds.extend(preds)
    return all_preds

print(f"Generating dev predictions...")
dev_preds = _get_preds(best_model, dev_df, tokenizer, MAX_LEN, DEVICE, 32, USE_AMP, AMP_DTYPE, THRESH)
print(f"Generating test predictions...")
test_preds = _get_preds(best_model, test_df, tokenizer, MAX_LEN, DEVICE, 32, USE_AMP, AMP_DTYPE, THRESH)

# Save predictions
(OUTPUT_DIR / "dev.txt").write_text("\n".join(map(str, dev_preds)))
(OUTPUT_DIR / "test.txt").write_text("\n".join(map(str, test_preds)))

# Verify dev predictions against actual labels
from sklearn.metrics import f1_score, accuracy_score, classification_report

dev_labels = dev_df["label_bin"].astype(int).tolist()
print("\n" + "=" * 60)
print("DEV SET VERIFICATION (should match trained F1)")
print("=" * 60)
print(f"F1 Score: {f1_score(dev_labels, dev_preds, pos_label=1):.4f}")
print(f"Accuracy: {accuracy_score(dev_labels, dev_preds):.4f}")
print("\nClassification Report:")
print(classification_report(dev_labels, dev_preds, target_names=["Non-PCL", "PCL"]))
print(f"\nFiles saved to: {OUTPUT_DIR}")

Test set: 3832 paragraphs
Loading model from checkpoint: /home/joshua_killa/doc/y3/PCL-detection/experiments/token-level/best_model.pt


Loading weights: 100%|██████████| 25/25 [00:00<00:00, 443.48it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model from epoch 2 with F1=0.5797
Generating dev predictions...
Generating test predictions...

DEV SET VERIFICATION (should match trained F1)
F1 Score: 0.5797
Accuracy: 0.9169

Classification Report:
              precision    recall  f1-score   support

     Non-PCL       0.96      0.95      0.95      1895
         PCL       0.56      0.60      0.58       199

    accuracy                           0.92      2094
   macro avg       0.76      0.78      0.77      2094
weighted avg       0.92      0.92      0.92      2094


Files saved to: /home/joshua_killa/doc/y3/PCL-detection/experiments/token-level


# 5-Fold Cross-Validation

Evaluate model generalization with 5-fold CV on training set using best hyperparameters.

In [ ]:
# ============================================================
# 5-FOLD CROSS-VALIDATION ON TRAINING SET
# ============================================================
# Uses best hyperparameters from Optuna study
# Trains for 7 epochs on each fold, picks best epoch F1

from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Dataset


class TensorDictDataset(Dataset):
    """Simple dataset wrapper for a dict of tensors (for CV fold subsets)."""
    def __init__(self, tensor_dict):
        self.data = tensor_dict
        # Get length from first tensor
        self._len = len(next(iter(tensor_dict.values())))
    
    def __len__(self):
        return self._len
    
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.data.items()}


def _extract_tensors_from_dataset(dataset):
    """Extract all tensors from TensorCacheDataset as a dict."""
    tensors = {
        "input_ids": dataset.input_ids,
        "attention_mask": dataset.attention_mask,
        "token_labels": dataset.token_labels,
        "token_loss_mask": dataset.token_loss_mask,
        "paragraph_label": dataset.paragraph_label,
    }
    if dataset.token_type_ids is not None:
        tensors["token_type_ids"] = dataset.token_type_ids
    return tensors


def run_5fold_cv(
    train_df,
    train_dataset,
    tokenizer,
    model_name,
    alpha_neg,
    lambda_token,
    lr,
    token_only_epochs,
    weight_decay,
    batch_size,
    grad_accum,
    epochs,
    device,
    use_amp,
    amp_dtype,
    needs_scaler,
    seed=42,
):
    """
    Run 5-fold cross-validation on training set.
    Returns list of best F1 scores for each fold and mean.
    """
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    
    labels = train_df["label_bin"].astype(int).to_numpy()
    indices = np.arange(len(train_df))
    
    # Extract all tensors once (avoid repeated attribute access)
    all_tensors = _extract_tensors_from_dataset(train_dataset)
    
    fold_results = []
    
    for fold_idx, (train_indices, val_indices) in enumerate(skf.split(indices, labels)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold_idx + 1}/5")
        print(f"Train: {len(train_indices)} | Val: {len(val_indices)}")
        print(f"{'='*60}")
        
        # Reset seed for each fold
        set_seed(seed)
        
        # Create subset datasets using indices
        fold_train_data = {k: v[train_indices] for k, v in all_tensors.items()}
        fold_val_data = {k: v[val_indices] for k, v in all_tensors.items()}
        
        fold_train_dataset = TensorDictDataset(fold_train_data)
        fold_val_dataset = TensorDictDataset(fold_val_data)
        
        # Balanced sampler for this fold
        fold_labels = labels[train_indices]
        fold_pos = int(fold_labels.sum())
        fold_neg = len(fold_labels) - fold_pos
        w_pos = 1.0 / max(fold_pos, 1)
        w_neg = 1.0 / max(fold_neg, 1)
        fold_weights = torch.tensor([w_pos if y == 1 else w_neg for y in fold_labels], dtype=torch.double)
        
        sampler = WeightedRandomSampler(fold_weights, num_samples=len(fold_weights), replacement=True)
        fold_train_loader = DataLoader(fold_train_dataset, batch_size=batch_size, sampler=sampler)
        fold_val_loader = DataLoader(fold_val_dataset, batch_size=64, shuffle=False)
        
        # Initialize model
        model = TokenCLSModel(
            model_name,
            lambda_token=lambda_token,
            pool_mode="max",
            alpha_neg=alpha_neg,
        ).to(device)
        
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        total_update_steps = (len(fold_train_loader) * epochs + grad_accum - 1) // grad_accum
        warmup_steps = int(0.07 * total_update_steps)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_update_steps,
        )
        
        scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and needs_scaler))
        
        best_val_f1 = -1.0
        best_epoch = -1
        
        for epoch in range(epochs):
            token_only = epoch < token_only_epochs
            
            # Curriculum
            if token_only:
                for p in model.paragraph_head.parameters():
                    p.requires_grad = False
                model.lambda_token = 1.0
            else:
                for p in model.paragraph_head.parameters():
                    p.requires_grad = True
                model.lambda_token = float(lambda_token)
            
            model.train()
            optimizer.zero_grad(set_to_none=True)
            
            last_step = 0
            for step, batch in enumerate(fold_train_loader, start=1):
                last_step = step
                batch = {k: v.to(device) for k, v in batch.items()}
                if token_only:
                    batch["paragraph_label"] = None
                
                if use_amp:
                    with torch.autocast(device_type="cuda", dtype=amp_dtype):
                        out = model(**batch)
                        loss = out["loss"]
                else:
                    out = model(**batch)
                    loss = out["loss"]
                
                loss = loss / grad_accum
                
                if scaler.is_enabled():
                    scaler.scale(loss).backward()
                else:
                    loss.backward()
                
                if (step % grad_accum) == 0:
                    if scaler.is_enabled():
                        scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    
                    if scaler.is_enabled():
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        optimizer.step()
                    
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
            
            # Flush remainder
            if last_step and (last_step % grad_accum) != 0:
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                
                if scaler.is_enabled():
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            # Evaluate on validation fold
            val_stats = stats_on_loader(
                fold_val_loader, model, device, use_amp, amp_dtype,
                threshold=0.5, compute_token_loss=False, compute_token_metrics=False, limit_batches=None
            )
            val_f1 = float(val_stats["f1"])
            
            print(f"  Epoch {epoch} | val_f1={val_f1:.4f} | val_loss={val_stats['loss']:.4f}")
            
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_epoch = epoch
        
        print(f"Fold {fold_idx + 1} BEST: epoch={best_epoch}, F1={best_val_f1:.4f}")
        fold_results.append({
            "fold": fold_idx + 1,
            "best_epoch": best_epoch,
            "best_f1": best_val_f1,
        })
        
        # Cleanup
        del model, optimizer, scheduler, scaler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return fold_results


# Load hyperparameters from Optuna study (standalone - doesn't require cell 20)
best_trial = study.best_trial
ALPHA_NEG = best_trial.params["alpha_neg"]
LAMBDA_TOKEN = best_trial.params["lambda_token"]
LR = best_trial.params["lr"]
TOKEN_ONLY_EPOCHS = best_trial.params["token_only_epochs"]
WEIGHT_DECAY = best_trial.params["weight_decay"]
BATCH_SIZE = best_trial.user_attrs.get("batch_size", 16)
GRAD_ACCUM = best_trial.user_attrs.get("grad_accum", 2)
SEED = 42

# Run 5-fold CV using best hyperparameters
print("Running 5-Fold Cross-Validation with best hyperparameters...")
print(f"alpha_neg={ALPHA_NEG:.4f}, lambda_token={LAMBDA_TOKEN:.4f}, lr={LR:.2e}")
print(f"token_only_epochs={TOKEN_ONLY_EPOCHS}, weight_decay={WEIGHT_DECAY:.4f}")

cv_results = run_5fold_cv(
    train_df=train_df,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    model_name=MODEL_NAME,
    alpha_neg=ALPHA_NEG,
    lambda_token=LAMBDA_TOKEN,
    lr=LR,
    token_only_epochs=TOKEN_ONLY_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    epochs=7,  # Always train for 7 epochs
    device=DEVICE,
    use_amp=USE_AMP,
    amp_dtype=AMP_DTYPE,
    needs_scaler=NEEDS_SCALER,
    seed=SEED,
)

# Summary
print("\n" + "=" * 60)
print("5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 60)
for r in cv_results:
    print(f"Fold {r['fold']}: best_epoch={r['best_epoch']}, F1={r['best_f1']:.4f}")

mean_f1 = np.mean([r["best_f1"] for r in cv_results])
std_f1 = np.std([r["best_f1"] for r in cv_results])
print(f"\nMean F1: {mean_f1:.4f} ± {std_f1:.4f}")
print("=" * 60)

# Save CV results
cv_results_path = ROOT / "experiments" / "token-level" / "cv_results.txt"
with open(cv_results_path, "w") as f:
    f.write("5-Fold Cross-Validation Results\n")
    f.write("=" * 40 + "\n")
    f.write(f"alpha_neg={ALPHA_NEG:.4f}\n")
    f.write(f"lambda_token={LAMBDA_TOKEN:.4f}\n")
    f.write(f"lr={LR:.2e}\n")
    f.write(f"token_only_epochs={TOKEN_ONLY_EPOCHS}\n")
    f.write(f"weight_decay={WEIGHT_DECAY:.4f}\n")
    f.write("=" * 40 + "\n")
    for r in cv_results:
        f.write(f"Fold {r['fold']}: epoch={r['best_epoch']}, F1={r['best_f1']:.4f}\n")
    f.write("=" * 40 + "\n")
    f.write(f"Mean F1: {mean_f1:.4f} ± {std_f1:.4f}\n")
print(f"CV results saved to: {cv_results_path}")

Running 5-Fold Cross-Validation with best hyperparameters...
alpha_neg=0.5331, lambda_token=0.6327, lr=1.56e-05
token_only_epochs=1, weight_decay=0.0074

FOLD 1/5
Train: 6700 | Val: 1675


Loading weights: 100%|██████████| 25/25 [00:00<00:00, 451.77it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_1080127/64931946.py:118: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and needs_scaler))


  Epoch 0 | val_f1=0.0000 | val_loss=0.4875
  Epoch 1 | val_f1=0.4989 | val_loss=0.3836
  Epoch 2 | val_f1=0.0000 | val_loss=0.5519
  Epoch 3 | val_f1=0.1724 | val_loss=0.7342
  Epoch 4 | val_f1=0.1724 | val_loss=0.7506
  Epoch 5 | val_f1=0.1724 | val_loss=0.7511
  Epoch 6 | val_f1=0.1734 | val_loss=0.7104
Fold 1 BEST: epoch=1, F1=0.4989

FOLD 2/5
Train: 6700 | Val: 1675


Loading weights: 100%|██████████| 25/25 [00:00<00:00, 460.69it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_1080127/64931946.py:118: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and needs_scaler))


  Epoch 0 | val_f1=0.0000 | val_loss=0.4964
